In [3]:
# =============================================================================
#  INDIA-PRIMARY DATA PREPARATION PIPELINE (Single Cell)
#  Downloads 11 datasets → harmonizes → cleans → engineers features → saves
#  NO training. Training code loads from prepared_data/*.parquet later.
#  Target: 150K+ rows | Kaggle T4 x2 | India-primary with global augmentation
# =============================================================================

import subprocess, sys, os, zipfile, shutil, warnings, json, time, glob, re
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)

# =============================================================================
# SECTION 1: INSTALL (only what's needed for data prep)
# =============================================================================
print("=" * 70)
print("📦 SECTION 1: INSTALL DEPENDENCIES")
print("=" * 70)

def pip_install(pkgs):
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs)
        return True
    except Exception as e:
        print(f"   ⚠️ pip failed: {str(e)[:120]}")
        return False

pip_install(['kaggle', 'pyarrow', 'scipy>=1.10.0,<1.17', 'scikit-learn>=1.3.0,<1.9.0'])

# =============================================================================
# SECTION 2: SETUP & IMPORTS
# =============================================================================
print("\n" + "=" * 70)
print("📚 SECTION 2: IMPORTS & SETUP")
print("=" * 70)

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder

WORK_DIR = '/kaggle/working'
DATA_DIR = os.path.join(WORK_DIR, 'data')
PREP_DIR = os.path.join(WORK_DIR, 'prepared_data')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PREP_DIR, exist_ok=True)

def run_cmd(cmd, desc):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"   ⚠️ {desc}: {r.stderr.strip()[:120]}")
        return False
    return True

def find_col(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

# =============================================================================
# SECTION 3: KAGGLE API SETUP
# =============================================================================
print("\n" + "=" * 70)
print("🔑 SECTION 3: KAGGLE API")
print("=" * 70)

if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    if os.path.exists('kaggle.json'):
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
        shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
        os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
        print("✅ Kaggle API configured")
    else:
        print("⚠️ kaggle.json not found — Kaggle downloads will be skipped")

# =============================================================================
# SECTION 4: DOWNLOAD ALL DATASETS (auto-verify, continue on failure)
# =============================================================================
print("\n" + "=" * 70)
print("📥 SECTION 4: DOWNLOADING ALL 11 DATASETS")
print("=" * 70)

kaggle_datasets = [
    # (slug, label)
    ('s3programmer/vehcle-emission-dataset', 'India Vehicle Emission'),
    ('ahmettyilmazz/fuel-consumption', 'Kaggle Canada 2000-2022'),
    ('sumangoudakaggle/global-car-specs-and-technical-data', 'Global Car Specs'),
    ('iconicwasil/vehicle-specs-and-carbon-footprint-2015-2026', 'Vehicle Specs 2015-2026'),
    ('debajyotipodder/co2-emission-by-vehicles', 'CO2 Emission by Vehicles'),
    ('brsahan/vehicle-co2-emissions-dataset', 'Vehicle CO2 Emissions'),
    ('reubenowenwilliams/vehicle-emissions-dataset', 'UK VCA'),
]
for slug, label in kaggle_datasets:
    print(f"\n   Downloading: {label}")
    run_cmd(f"kaggle datasets download -d {slug} -p {DATA_DIR} --unzip -q", label)

# EPA direct download
print("\n   Downloading: EPA Fuel Economy")
EPA_ZIP = os.path.join(DATA_DIR, 'epa_vehicles.csv.zip')
if run_cmd(f"wget -q -O {EPA_ZIP} "
           f"https://www.fueleconomy.gov/feg/epadata/vehicles.csv.zip", "EPA"):
    run_cmd(f"unzip -q -o {EPA_ZIP} -d {DATA_DIR}", "EPA unzip")
    print("   ✅ EPA downloaded")

# Open Canada multi-year files
print("\n   Downloading: Open Canada Fuel Ratings")
try:
    import urllib.request
    ckan_url = ("https://open.canada.ca/data/api/3/action/package_show?"
                "id=98f1a129-f628-4ce4-b24d-6f16bf24dd64")
    with urllib.request.urlopen(ckan_url, timeout=30) as resp:
        pkg = json.loads(resp.read())
    downloaded = []
    for res in pkg['result']['resources']:
        name = str(res.get('name', ''))
        if res.get('format') == 'CSV':
            fname = None
            if '2015-2024' in name: fname = 'canada_2015_2024.csv'
            elif '1995-2014' in name: fname = 'canada_1995_2014.csv'
            elif '2026' in name: fname = 'canada_2026.csv'
            if fname and not os.path.exists(os.path.join(DATA_DIR, fname)):
                try:
                    urllib.request.urlretrieve(res['url'], os.path.join(DATA_DIR, fname))
                    downloaded.append(fname)
                except Exception:
                    pass
    print(f"   ✅ Downloaded: {downloaded}")
except Exception as e:
    print(f"   ⚠️ Open Canada: {str(e)[:120]}")

# Show all downloaded files
print("\n📂 Files in data/:")
for f in sorted(os.listdir(DATA_DIR)):
    fp = os.path.join(DATA_DIR, f)
    if os.path.isfile(fp):
        size_mb = os.path.getsize(fp) / 1024 / 1024
        print(f"   {f}: {size_mb:.2f} MB")

# =============================================================================
# SECTION 5: FEATURE ENGINEERING HELPERS
# =============================================================================
print("\n" + "=" * 70)
print("🔧 SECTION 5: HARMONIZATION HELPERS")
print("=" * 70)

def parse_transmission(t):
    if pd.isna(t):
        return pd.Series({'gear_count': 0, 'is_automatic': 0, 'is_cvt': 0})
    t = str(t).upper()
    is_cvt = int('CVT' in t or 'VARIABLE' in t)
    is_auto = int('A' in t[:3] or 'AUTO' in t or is_cvt)
    m = re.search(r'(\d+)', t)
    return pd.Series({'gear_count': int(m.group(1)) if m else 0,
                      'is_automatic': is_auto, 'is_cvt': is_cvt})

def unify_vehicle_class(vc):
    if pd.isna(vc): return 'Unknown'
    v = str(vc).strip().upper()
    if v == 'MOTORCYCLE' or v == 'TWO WHEELER': return 'Motorcycle'
    if v == 'BUS': return 'Bus'
    if v == 'TRUCK' or 'PICKUP' in v: return 'Truck'
    if v == 'CAR': return 'Car'
    if any(k in v for k in ['SUBCOMPACT','COMPACT','MINICOMPACT']): return 'Car'
    if any(k in v for k in ['MID-SIZE','MIDSIZE','FULL-SIZE','LARGE']): return 'Car'
    if any(k in v for k in ['SPORT','LUXURY','STATION WAGON']): return 'Car'
    if any(k in v for k in ['SUV','SPORT UTILITY','CROSSOVER']): return 'SUV'
    if any(k in v for k in ['MINIVAN','VAN']): return 'Van'
    return 'Other'

def unify_fuel_type(ft):
    if pd.isna(ft): return 'Unknown'
    f = str(ft).strip().upper()
    if f == 'X': return 'Petrol'
    if f == 'Z': return 'Premium'
    if f == 'D': return 'Diesel'
    if f == 'E': return 'E85'
    if f == 'N': return 'CNG'
    if 'PETROL' in f or 'REGULAR' in f or 'GASOLINE' in f: return 'Petrol'
    if 'PREMIUM' in f: return 'Premium'
    if 'DIESEL' in f: return 'Diesel'
    if 'ELECTRIC' in f or f == 'EV': return 'Electric'
    if 'HYBRID' in f: return 'Hybrid'
    if 'E85' in f: return 'E85'
    if 'CNG' in f or 'NATURAL GAS' in f: return 'CNG'
    return 'Other'

def parse_drivetrain(row):
    model = str(row.get('model', '') if hasattr(row, 'get') else '')
    drive = str(row.get('drive', '') if hasattr(row, 'get') else '')
    text = (model + ' ' + drive).upper()
    if any(k in text for k in ['AWD','4WD','ALL WHEEL','4MOTION','XDRIVE','QUATTRO']): return 'AWD'
    if any(k in text for k in ['RWD','REAR']): return 'RWD'
    if any(k in text for k in ['FWD','FRONT','2WD']): return 'FWD'
    return 'UNK'

def parse_turbo(row):
    text = str(row.get('model', '') if hasattr(row, 'get') else '').upper()
    return int(any(k in text for k in
                   ['TURBO','TDI','TSI','ECOBOOST','TFSI','GTI','STI','TURBODIESEL']))

def oof_target_encode(train_df, val_df, test_df, col, target, n_splits=5, smoothing=30):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_enc = np.zeros(len(train_df))
    gm = train_df[target].mean()
    for tr_idx, vl_idx in kf.split(train_df):
        g = train_df.iloc[tr_idx].groupby(col)[target].agg(['mean', 'count'])
        g['enc'] = (g['mean'] * g['count'] + gm * smoothing) / (g['count'] + smoothing)
        train_enc[vl_idx] = train_df.iloc[vl_idx][col].map(g['enc'].to_dict()).fillna(gm).values
    g = train_df.groupby(col)[target].agg(['mean', 'count'])
    g['enc'] = (g['mean'] * g['count'] + gm * smoothing) / (g['count'] + smoothing)
    m = g['enc'].to_dict()
    return (train_enc,
            val_df[col].map(m).fillna(gm).values,
            test_df[col].map(m).fillna(gm).values)

# =============================================================================
# SECTION 6: HARMONIZE ALL DATASETS (India-aware schema)
# =============================================================================
print("\n" + "=" * 70)
print("🔗 SECTION 6: HARMONIZING ALL DATASETS")
print("=" * 70)

all_frames = []
ALL_COLS = ['year','make','model','vehicle_class','vehicle_class_unified',
            'fuel_type','fuel_type_unified','engine_size_l','cylinders',
            'transmission','vehicle_age','mileage_km','speed_kmh','acceleration_ms2',
            'road_type','traffic_condition','temperature_c','humidity_pct',
            'wind_speed_ms','air_pressure_hpa','power_hp','drive',
            'co2_g_per_km','fuel_consumption_l_per_100km',
            'region','regulatory_cycle','data_source','is_india']

def add_if_valid(df, name):
    if df is None or len(df) == 0:
        print(f"   ⚠️ {name}: empty"); return
    if 'co2_g_per_km' not in df.columns:
        print(f"   ⚠️ {name}: no co2_g_per_km"); return
    df = df.dropna(subset=['co2_g_per_km'])
    df = df[df['co2_g_per_km'] > 0]
    if len(df) == 0:
        print(f"   ⚠️ {name}: 0 valid CO2 rows"); return
    for c in ALL_COLS:
        if c not in df.columns: df[c] = np.nan
    all_frames.append(df[ALL_COLS])
    print(f"   ✅ {name}: {df.shape}")

# --- 6a. India Vehicle Emission (PRIMARY) ---
try:
    m = glob.glob(os.path.join(DATA_DIR, '**/vehicle_emission_dataset.csv'), recursive=True)
    if m:
        df_i = pd.read_csv(m[0], encoding='latin-1')
        df_i.columns = [c.lower().replace(' ','_') for c in df_i.columns]
        rmap = {'vehicle_type':'vehicle_class','engine_size':'engine_size_l',
                'age_of_vehicle':'vehicle_age','mileage':'mileage_km',
                'speed':'speed_kmh','acceleration':'acceleration_ms2',
                'road_type':'road_type','traffic_conditions':'traffic_condition',
                'temperature':'temperature_c','humidity':'humidity_pct',
                'wind_speed':'wind_speed_ms','air_pressure':'air_pressure_hpa',
                'co2_emissions':'co2_g_per_km'}
        df_i = df_i.rename(columns={k:v for k,v in rmap.items() if k in df_i.columns})
        df_i['region']='India'; df_i['regulatory_cycle']='ARAI'
        df_i['data_source']='India_Vehicle_Emission'; df_i['is_india']=1
        df_i['vehicle_class_unified'] = df_i['vehicle_class'].apply(unify_vehicle_class)
        df_i['fuel_type_unified'] = df_i['fuel_type'].apply(unify_fuel_type)
        add_if_valid(df_i, 'India Vehicle Emission')
except Exception as e:
    print(f"   ⚠️ India: {e}")

# --- 6b. Kaggle Canada ---
try:
    m = glob.glob(os.path.join(DATA_DIR, '**/Fuel_Consumption_2000-2022.csv'), recursive=True)
    if m:
        df_k = pd.read_csv(m[0], encoding='latin-1').rename(columns={
            'YEAR':'year','MAKE':'make','MODEL':'model','VEHICLE CLASS':'vehicle_class',
            'ENGINE SIZE':'engine_size_l','CYLINDERS':'cylinders','TRANSMISSION':'transmission',
            'FUEL':'fuel_type','FUEL CONSUMPTION':'fuel_consumption_l_per_100km',
            'EMISSIONS':'co2_g_per_km'})
        df_k = df_k.drop(columns=[c for c in ['HWY (L/100 km)','COMB (L/100 km)','COMB (mpg)']
                                  if c in df_k.columns], errors='ignore')
        df_k['region']='Canada'; df_k['regulatory_cycle']='NEDC'
        df_k['data_source']='Kaggle_Canada'; df_k['is_india']=0
        df_k['vehicle_class_unified'] = df_k['vehicle_class'].apply(unify_vehicle_class)
        df_k['fuel_type_unified'] = df_k['fuel_type'].apply(unify_fuel_type)
        add_if_valid(df_k, 'Kaggle Canada')
except Exception as e:
    print(f"   ⚠️ Kaggle Canada: {e}")

# --- 6c. EPA ---
try:
    m = glob.glob(os.path.join(DATA_DIR, '**/vehicles.csv'), recursive=True)
    if m:
        df_e = pd.read_csv(m[0], low_memory=False)
        emap = {'year':'year','make':'make','model':'model','VClass':'vehicle_class',
                'displ':'engine_size_l','cylinders':'cylinders','trany':'transmission',
                'fuelType':'fuel_type','co2TailpipeGpm':'co2_g_per_km',
                'comb08':'fuel_consumption_mpg','drive':'drive','hlv':'power_hp'}
        df_e = df_e[[c for c in emap if c in df_e.columns]].rename(columns=emap)
        if 'fuel_consumption_mpg' in df_e.columns:
            df_e['fuel_consumption_l_per_100km'] = 235.21/df_e['fuel_consumption_mpg'].replace(0,np.nan)
            df_e = df_e.drop(columns=['fuel_consumption_mpg'])
        df_e['region']='US'; df_e['regulatory_cycle']='EPA'
        df_e['data_source']='EPA_FuelEconomy'; df_e['is_india']=0
        df_e = df_e[df_e['co2_g_per_km']>0]
        df_e['vehicle_class_unified'] = df_e['vehicle_class'].apply(unify_vehicle_class)
        df_e['fuel_type_unified'] = df_e['fuel_type'].apply(unify_fuel_type)
        add_if_valid(df_e, 'EPA')
except Exception as e:
    print(f"   ⚠️ EPA: {e}")

# --- 6d/e/f. Open Canada files ---
def load_canada(fname, source_label):
    f = os.path.join(DATA_DIR, fname)
    if not os.path.exists(f):
        print(f"   ⚠️ {source_label}: file not found")
        return None
    try:
        df_c = pd.read_csv(f, encoding='latin-1')
        col_map = {}
        for target, cands in {
            'year': ['Annee modele','Model year','Year'],
            'make': ['Marque','Make'],
            'model': ['Modele','Model'],
            'vehicle_class': ['Categorie de vehicule','Vehicle class'],
            'engine_size_l': ['Cylindree (L)','Engine size (L)'],
            'cylinders': ['Cylindres','Cylinders'],
            'transmission': ['Transmission'],
            'fuel_type': ['Type de carburant','Fuel type'],
            'co2_g_per_km': ['Emissions de CO2 (g/km)','CO2 emissions (g/km)']
        }.items():
            c = find_col(df_c, cands)
            if c: col_map[c] = target
        df_c = df_c.rename(columns=col_map)
        df_c = df_c.drop(columns=[c for c in df_c.columns
                                  if any(k in str(c) for k in ['Combinee','Ville','Route'])],
                         errors='ignore')
        df_c['region']='Canada'; df_c['regulatory_cycle']='NEDC'
        df_c['data_source']=source_label; df_c['is_india']=0
        df_c['vehicle_class_unified'] = df_c['vehicle_class'].apply(unify_vehicle_class)
        df_c['fuel_type_unified'] = df_c['fuel_type'].apply(unify_fuel_type)
        add_if_valid(df_c, source_label)
    except Exception as e:
        print(f"   ⚠️ {source_label}: {e}")

load_canada('canada_2015_2024.csv', 'Canada_2015_2024')
load_canada('canada_1995_2014.csv', 'Canada_1995_2014')
load_canada('canada_2026.csv', 'Canada_2026')

# --- 6g. Global Car Specs ---
try:
    for f in glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True):
        base = os.path.basename(f).lower()
        if 'global' in base or 'car_spec' in base or 'technical' in base:
            df_g = pd.read_csv(f, encoding='latin-1', low_memory=False)
            df_g.columns = [c.lower().replace(' ','_').replace('(','_').replace(')','_')
                           for c in df_g.columns]
            col_map = {}
            for target, cands in {
                'year': ['year','production_start','model_year'],
                'make': ['brand','make','manufacturer'],
                'model': ['model','modification'],
                'engine_size_l': ['engine_displacement','displacement','engine_capacity'],
                'cylinders': ['cylinders','number_of_cylinders'],
                'power_hp': ['power_hp','power','horsepower'],
                'fuel_type': ['fuel_type','fuel'],
                'co2_g_per_km': ['co2_emissions','co2','co2_g_km']
            }.items():
                c = find_col(df_g, cands)
                if c: col_map[c] = target
            df_g = df_g.rename(columns=col_map)
            if 'co2_g_per_km' in df_g.columns:
                df_g['region']='Global'; df_g['regulatory_cycle']='Mixed'
                df_g['data_source']='Global_Car_Specs'; df_g['is_india']=0
                if 'vehicle_class' in df_g.columns:
                    df_g['vehicle_class_unified'] = df_g['vehicle_class'].apply(unify_vehicle_class)
                else:
                    df_g['vehicle_class_unified'] = 'Car'
                if 'fuel_type' in df_g.columns:
                    df_g['fuel_type_unified'] = df_g['fuel_type'].apply(unify_fuel_type)
                else:
                    df_g['fuel_type_unified'] = 'Petrol'
                add_if_valid(df_g, 'Global Car Specs')
            break
except Exception as e:
    print(f"   ⚠️ Global Car Specs: {e}")

# --- 6h. Vehicle Specs 2015-2026 ---
try:
    for f in glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True):
        base = os.path.basename(f).lower()
        if 'ev_vs_ice' in base or 'vehicle_spec' in base:
            df_v = pd.read_csv(f, encoding='latin-1')
            df_v.columns = [c.lower().replace(' ','_') for c in df_v.columns]
            rmap = {'make':'make','model':'model','year':'year',
                    'engine_cylinders':'cylinders','engine_size_l':'engine_size_l',
                    'drivetrain':'drive','transmission':'transmission',
                    'fuel_type':'fuel_type','combined_mpg':'fuel_consumption_mpg',
                    'co2_emissions_g_per_mile':'co2_g_per_km',
                    'vehicle_category':'vehicle_class'}
            df_v = df_v.rename(columns={k:v for k,v in rmap.items() if k in df_v.columns})
            if 'co2_g_per_km' in df_v.columns:
                df_v['co2_g_per_km'] = df_v['co2_g_per_km'] / 1.60934
            if 'fuel_consumption_mpg' in df_v.columns:
                df_v['fuel_consumption_l_per_100km'] = 235.21/df_v['fuel_consumption_mpg'].replace(0,np.nan)
                df_v = df_v.drop(columns=['fuel_consumption_mpg'])
            df_v['region']='Global'; df_v['regulatory_cycle']='EPA'
            df_v['data_source']='Vehicle_Specs_2026'; df_v['is_india']=0
            if 'vehicle_class' in df_v.columns:
                df_v['vehicle_class_unified'] = df_v['vehicle_class'].apply(unify_vehicle_class)
            else:
                df_v['vehicle_class_unified'] = 'Car'
            if 'fuel_type' in df_v.columns:
                df_v['fuel_type_unified'] = df_v['fuel_type'].apply(unify_fuel_type)
            else:
                df_v['fuel_type_unified'] = 'Petrol'
            add_if_valid(df_v, 'Vehicle Specs 2015-2026')
            break
except Exception as e:
    print(f"   ⚠️ Vehicle Specs 2026: {e}")

# --- 6i. CO2 Emission by Vehicles ---
try:
    m = glob.glob(os.path.join(DATA_DIR, '**/CO2_Emissions_Canada.csv'), recursive=True)
    if m:
        df_co2 = pd.read_csv(m[0], encoding='latin-1')
        df_co2.columns = [c.lower().replace(' ','_') for c in df_co2.columns]
        co2_col = find_col(df_co2, ['co2_emissions','co2_emissions(g/km)'])
        eng_col = find_col(df_co2, ['engine_size','engine_size(l)'])
        fuel_col = find_col(df_co2, ['fuel_consumption_comb_(l/100_km)'])
        rmap = {}
        if co2_col: rmap[co2_col] = 'co2_g_per_km'
        if eng_col: rmap[eng_col] = 'engine_size_l'
        if fuel_col: rmap[fuel_col] = 'fuel_consumption_l_per_100km'
        df_co2 = df_co2.rename(columns=rmap)
        df_co2['region']='Canada'; df_co2['regulatory_cycle']='NEDC'
        df_co2['data_source']='CO2_Canada'; df_co2['is_india']=0
        df_co2['year']=2014
        if 'vehicle_class' in df_co2.columns:
            df_co2['vehicle_class_unified'] = df_co2['vehicle_class'].apply(unify_vehicle_class)
        else:
            df_co2['vehicle_class_unified'] = 'Car'
        if 'fuel_type' in df_co2.columns:
            df_co2['fuel_type_unified'] = df_co2['fuel_type'].apply(unify_fuel_type)
        else:
            df_co2['fuel_type_unified'] = 'Petrol'
        add_if_valid(df_co2, 'CO2 Emission by Vehicles')
except Exception as e:
    print(f"   ⚠️ CO2 Vehicles: {e}")

# --- 6j. UK VCA ---
try:
    m = glob.glob(os.path.join(DATA_DIR, '**/vehicle_emissions*.csv'), recursive=True)
    if m:
        df_u = pd.read_csv(m[0], encoding='latin-1')
        df_u.columns = [c.lower().replace(' ','_').replace('(','_').replace(')','_')
                       for c in df_u.columns]
        co2_col = find_col(df_u, ['co2','co2_g_per_km','co2_emissions',
                                   'co2_g_km','co2_emissions_g_km'])
        if co2_col:
            df_u = df_u.rename(columns={co2_col: 'co2_g_per_km'})
        df_u['region']='UK'; df_u['regulatory_cycle']='WLTP'
        df_u['data_source']='UK_VCA'; df_u['is_india']=0
        if 'vehicle_class' in df_u.columns:
            df_u['vehicle_class_unified'] = df_u['vehicle_class'].apply(unify_vehicle_class)
        else:
            df_u['vehicle_class_unified'] = 'Car'
        if 'fuel_type' in df_u.columns:
            df_u['fuel_type_unified'] = df_u['fuel_type'].apply(unify_fuel_type)
        else:
            df_u['fuel_type_unified'] = 'Petrol'
        add_if_valid(df_u, 'UK VCA')
except Exception as e:
    print(f"   ⚠️ UK VCA: {e}")

# --- 6k. Vehicle CO2 Emissions (brsahan) ---
try:
    for f in glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True):
        base = os.path.basename(f).lower()
        if 'co2' in base and 'emission' in base and 'canada' not in base and 'vehicle' in base:
            df_vc = pd.read_csv(f, encoding='latin-1')
            df_vc.columns = [c.lower().replace(' ','_').replace('(','_').replace(')','_')
                            for c in df_vc.columns]
            co2_col = find_col(df_vc, ['co2_emissions','co2_emissions_g_km','co2','co2_g_per_km'])
            if co2_col:
                df_vc = df_vc.rename(columns={co2_col: 'co2_g_per_km'})
            df_vc['region']='Global'; df_vc['regulatory_cycle']='Mixed'
            df_vc['data_source']='Vehicle_CO2'; df_vc['is_india']=0
            df_vc['vehicle_class_unified'] = 'Car'
            if 'fuel_type' in df_vc.columns:
                df_vc['fuel_type_unified'] = df_vc['fuel_type'].apply(unify_fuel_type)
            else:
                df_vc['fuel_type_unified'] = 'Petrol'
            add_if_valid(df_vc, 'Vehicle CO2')
            break
except Exception as e:
    print(f"   ⚠️ Vehicle CO2: {e}")

# --- Final unification ---
if not all_frames:
    raise RuntimeError("No datasets loaded. Check Kaggle API key and internet.")

df_unified = pd.concat(all_frames, ignore_index=True)
print(f"\n📐 Unified shape: {df_unified.shape}")
print(f"📊 Sources:\n{df_unified['data_source'].value_counts().to_string()}")
print(f"🇮🇳 India rows: {(df_unified['is_india']==1).sum()}")

# =============================================================================
# SECTION 7: CLEANING
# =============================================================================
print("\n" + "=" * 70)
print("🧹 SECTION 7: CLEANING")
print("=" * 70)

df = df_unified.drop_duplicates().copy()

NUM_COLS = ['year','engine_size_l','cylinders','co2_g_per_km',
            'fuel_consumption_l_per_100km','power_hp','speed_kmh',
            'acceleration_ms2','vehicle_age','mileage_km','temperature_c',
            'humidity_pct','wind_speed_ms','air_pressure_hpa']
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df = df[df['co2_g_per_km'] > 0]
q999 = df['co2_g_per_km'].quantile(0.999)
before = len(df)
df = df[df['co2_g_per_km'] <= q999]
print(f"✅ Target trim: dropped {before-len(df)} extreme outliers (>{q999:.0f} g/km)")
print(f"✅ After cleaning: {df.shape}")
print(f"🇮🇳 India rows: {(df['is_india']==1).sum()}")

# =============================================================================
# SECTION 8: FEATURE ENGINEERING
# =============================================================================
print("\n" + "=" * 70)
print("🔧 SECTION 8: FEATURE ENGINEERING")
print("=" * 70)

# Missingness indicators (CRITICAL for India vs non-India)
df['has_speed']        = df['speed_kmh'].notna().astype(int)
df['has_acceleration'] = df['acceleration_ms2'].notna().astype(int)
df['has_weather']      = (df['temperature_c'].notna() & df['humidity_pct'].notna()).astype(int)
df['has_road']         = df['road_type'].notna().astype(int) if 'road_type' in df.columns else 0
df['has_age_mileage']  = (df['vehicle_age'].notna() & df['mileage_km'].notna()).astype(int)

# Categorical fills
CAT_FILL = ['make','model','vehicle_class','vehicle_class_unified','fuel_type',
            'fuel_type_unified','transmission','road_type','traffic_condition',
            'drive','region','regulatory_cycle','data_source']
for c in CAT_FILL:
    if c in df.columns:
        df[c] = df[c].fillna('UNKNOWN').astype(str)

# Base engineered
df['engine_per_cylinder'] = df['engine_size_l'] / df['cylinders'].replace(0,1)
df['engine_squared']      = df['engine_size_l'] ** 2
df['log_engine']          = np.log1p(df['engine_size_l'].clip(lower=0))
df['log_power']           = np.log1p(df['power_hp'].clip(lower=0))
df['age']                 = 2026 - df['year']
df['age_decay']           = np.exp(-df['age'].clip(lower=0)/10)
df['age_x_engine']        = df['age'] * df['engine_size_l']
df['cyl_x_engine']        = df['cylinders'] * df['engine_size_l']
df['engine_x_year']       = df['engine_size_l'] * df['year']
df['is_4cyl']             = (df['cylinders']==4).astype(int)
df['is_6cyl']             = (df['cylinders']==6).astype(int)
df['is_8cyl_plus']        = (df['cylinders']>=8).astype(int)
df['year_x_cyl']          = df['year'] * df['cylinders']

# India-specific interactions
df['speed_x_accel']    = df['speed_kmh'] * df['acceleration_ms2']
df['speed_x_engine']   = df['speed_kmh'] * df['engine_size_l']
df['age_x_mileage']    = df['vehicle_age'] * df['mileage_km']
df['temp_x_humidity']  = df['temperature_c'] * df['humidity_pct']
df['speed_per_engine'] = df['speed_kmh'] / (df['engine_size_l'] + 0.1)

# Transmission
trans = df['transmission'].apply(parse_transmission)
df = pd.concat([df, trans], axis=1)

# Drivetrain & turbo
df['drivetrain'] = df.apply(parse_drivetrain, axis=1)
df['turbo_flag'] = df.apply(parse_turbo, axis=1)

# Power features
df['power_to_weight']    = (df['power_hp'] /
                            (df['engine_size_l']*100 + 1)).replace([np.inf,-np.inf], 0)
df['power_per_cylinder'] = (df['power_hp'] /
                            df['cylinders'].replace(0,1)).replace([np.inf,-np.inf], 0)
df['hp_x_disp']          = df['power_hp'] * df['engine_size_l']

df['make_model'] = df['make'].astype(str) + '_' + df['model'].astype(str)

print(f"✅ Feature engineered: {df.shape}")

# =============================================================================
# SECTION 9: TRAIN/VAL/TEST SPLIT (stratified by source)
# =============================================================================
print("\n" + "=" * 70)
print("📊 SECTION 9: SPLIT")
print("=" * 70)

TARGET = 'co2_g_per_km'
df = df.reset_index(drop=True)
df['y_log'] = np.log1p(df[TARGET])

src_counts = df['data_source'].value_counts()
rare = src_counts[src_counts < 10].index.tolist()
if rare:
    df['data_source'] = df['data_source'].replace({s:'Other' for s in rare})

df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['data_source'])
df_train, df_val = train_test_split(
    df_train, test_size=0.15, random_state=42, stratify=df_train['data_source'])

df_train = df_train.copy(); df_val = df_val.copy(); df_test = df_test.copy()
print(f"📚 Train:{len(df_train)} | Val:{len(df_val)} | Test:{len(df_test)}")
india_test_mask = (df_test['data_source'].values == 'India_Vehicle_Emission')
print(f"🇮🇳 India test rows: {india_test_mask.sum()}")

# =============================================================================
# SECTION 10: OOF TARGET ENCODING
# =============================================================================
print("\n" + "=" * 70)
print("🎯 SECTION 10: OOF TARGET ENCODING")
print("=" * 70)

te_cols = ['make','model','make_model','vehicle_class','vehicle_class_unified',
           'fuel_type','fuel_type_unified','transmission','road_type',
           'traffic_condition','region','regulatory_cycle','data_source']
for col in te_cols:
    if col in df_train.columns:
        tr, vl, ts = oof_target_encode(df_train, df_val, df_test, col, 'y_log',
                                       n_splits=5, smoothing=30)
        df_train[f'{col}_te'] = tr; df_val[f'{col}_te'] = vl; df_test[f'{col}_te'] = ts
print(f"   ✅ {len(te_cols)} TE features added")

# Label-encode drivetrain
for col in ['drivetrain']:
    le = LabelEncoder()
    comb = pd.concat([df_train[col], df_val[col], df_test[col]]).astype(str)
    le.fit(comb)
    df_train[col+'_enc'] = le.transform(df_train[col].astype(str))
    df_val[col+'_enc']   = le.transform(df_val[col].astype(str))
    df_test[col+'_enc']  = le.transform(df_test[col].astype(str))

# =============================================================================
# SECTION 11: BUILD FEATURE MATRICES & SAVE
# =============================================================================
print("\n" + "=" * 70)
print("💾 SECTION 11: BUILDING & SAVING FEATURE MATRICES")
print("=" * 70)

DROP = [TARGET, 'y_log', 'fuel_consumption_l_per_100km',
        'make','model','make_model','vehicle_class','vehicle_class_unified',
        'fuel_type','fuel_type_unified','transmission','road_type',
        'traffic_condition','region','regulatory_cycle','data_source','drivetrain']
DROP = [c for c in DROP if c in df_train.columns]

feature_cols = [c for c in df_train.columns if c not in DROP
                and df_train[c].dtype != object]
print(f"🔢 Feature count: {len(feature_cols)}")

X_train_raw = df_train[feature_cols].replace([np.inf,-np.inf], np.nan)
X_val_raw   = df_val[feature_cols].replace([np.inf,-np.inf], np.nan)
X_test_raw  = df_test[feature_cols].replace([np.inf,-np.inf], np.nan)

# Imputed version
med = X_train_raw.median()
X_train_imp = X_train_raw.fillna(med).fillna(0)
X_val_imp   = X_val_raw.fillna(med).fillna(0)
X_test_imp  = X_test_raw.fillna(med).fillna(0)

# Targets
y_train_log  = df_train['y_log'].values
y_val_log    = df_val['y_log'].values
y_test_log   = df_test['y_log'].values
y_train_orig = df_train[TARGET].values
y_val_orig   = df_val[TARGET].values
y_test_orig  = df_test[TARGET].values

# Save to parquet
print("\n💾 Saving to parquet...")
X_train_raw.to_parquet(os.path.join(PREP_DIR, 'X_train_raw.parquet'))
X_val_raw.to_parquet(os.path.join(PREP_DIR, 'X_val_raw.parquet'))
X_test_raw.to_parquet(os.path.join(PREP_DIR, 'X_test_raw.parquet'))
X_train_imp.to_parquet(os.path.join(PREP_DIR, 'X_train_imp.parquet'))
X_val_imp.to_parquet(os.path.join(PREP_DIR, 'X_val_imp.parquet'))
X_test_imp.to_parquet(os.path.join(PREP_DIR, 'X_test_imp.parquet'))

np.save(os.path.join(PREP_DIR, 'y_train_log.npy'), y_train_log)
np.save(os.path.join(PREP_DIR, 'y_val_log.npy'), y_val_log)
np.save(os.path.join(PREP_DIR, 'y_test_log.npy'), y_test_log)
np.save(os.path.join(PREP_DIR, 'y_train_orig.npy'), y_train_orig)
np.save(os.path.join(PREP_DIR, 'y_val_orig.npy'), y_val_orig)
np.save(os.path.join(PREP_DIR, 'y_test_orig.npy'), y_test_orig)
np.save(os.path.join(PREP_DIR, 'india_test_mask.npy'), india_test_mask)

# Save the full cleaned dataset (for per-source diagnostics in training)
df.to_parquet(os.path.join(PREP_DIR, 'full_dataset.parquet'))
df_test[['data_source']].to_parquet(os.path.join(PREP_DIR, 'test_sources.parquet'))

# Save feature names, median, metadata
import pickle
with open(os.path.join(PREP_DIR, 'feature_cols.json'), 'w') as f:
    json.dump(feature_cols, f, indent=2)
with open(os.path.join(PREP_DIR, 'feature_median.pkl'), 'wb') as f:
    pickle.dump(med, f)

metadata = {
    'total_rows': int(len(df)),
    'train_rows': int(len(df_train)),
    'val_rows': int(len(df_val)),
    'test_rows': int(len(df_test)),
    'india_rows_total': int((df['is_india']==1).sum()),
    'india_rows_test': int(india_test_mask.sum()),
    'n_features': int(len(feature_cols)),
    'sources': df['data_source'].value_counts().to_dict(),
    'target': TARGET,
    'target_transform': 'log1p',
    'split_seed': 42,
    'feature_list': feature_cols,
}
with open(os.path.join(PREP_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Files saved to {PREP_DIR}/")
for f in sorted(os.listdir(PREP_DIR)):
    fp = os.path.join(PREP_DIR, f)
    size_kb = os.path.getsize(fp) / 1024
    print(f"   {f}: {size_kb:.1f} KB")

# =============================================================================
# SECTION 12: SUMMARY & ZIP
# =============================================================================
print("\n" + "=" * 70)
print("📊 SECTION 12: DATA PREPARATION SUMMARY")
print("=" * 70)

print(f"""
✅ Total rows loaded       : {len(df):,}
✅ Total columns (raw)     : {df.shape[1]}
✅ Features for training   : {len(feature_cols)}
✅ Train / Val / Test      : {len(df_train):,} / {len(df_val):,} / {len(df_test):,}
🇮🇳 India rows (total)     : {(df['is_india']==1).sum():,}
🇮🇳 India rows (test)      : {india_test_mask.sum():,}

📊 Dataset sources:
{df['data_source'].value_counts().to_string()}

📁 Prepared files saved to: {PREP_DIR}/
""")

# Create ZIP with prepared data only (skip raw downloads to keep size small)
ZIP_NAME = 'prepared_dataset.zip'
ZIP_PATH = os.path.join(WORK_DIR, ZIP_NAME)
if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)

print(f"\n📦 Creating ZIP: {ZIP_NAME}")
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(PREP_DIR):
        fp = os.path.join(PREP_DIR, f)
        zf.write(fp, os.path.join('prepared_data', f))

size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"✅ ZIP created: {ZIP_PATH} ({size_mb:.2f} MB)")

# Display download link
from IPython.display import FileLink, display
print("\n📥 Download prepared data:")
display(FileLink(ZIP_NAME))

print("\n" + "=" * 70)
print("🎉 DATA PREPARATION COMPLETE")
print("=" * 70)
print("""
Next step: run the TRAINING cell. It will load from:
  /kaggle/working/prepared_data/X_train_raw.parquet
  /kaggle/working/prepared_data/y_train_log.npy
  ...etc

Ask for the training code when you're ready.
""")

📦 SECTION 1: INSTALL DEPENDENCIES
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 15.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.



📚 SECTION 2: IMPORTS & SETUP

🔑 SECTION 3: KAGGLE API
⚠️ kaggle.json not found — Kaggle downloads will be skipped

📥 SECTION 4: DOWNLOADING ALL 11 DATASETS

   Downloading: India Vehicle Emission

   Downloading: Kaggle Canada 2000-2022

   Downloading: Global Car Specs

   Downloading: Vehicle Specs 2015-2026

   Downloading: CO2 Emission by Vehicles

   Downloading: Vehicle CO2 Emissions

   Downloading: UK VCA

   Downloading: EPA Fuel Economy
   ✅ EPA downloaded

   Downloading: Open Canada Fuel Ratings
   ✅ Downloaded: ['canada_2026.csv', 'canada_2015_2024.csv', 'canada_1995_2014.csv']

📂 Files in data/:
   212.csv: 0.00 MB
   AC.csv: 0.01 MB
   AITO.csv: 0.03 MB
   Abarth.csv: 0.06 MB
   Acura.csv: 0.12 MB
   Aeolus.csv: 0.04 MB
   Aion.csv: 0.05 MB
   Aistaland.csv: 0.00 MB
   Aiways.csv: 0.00 MB
   Aixam.csv: 0.02 MB
   Alfa_Romeo.csv: 0.46 MB
   Alpina.csv: 0.17 MB
   Alpine.csv: 0.02 MB
   Anfini.csv: 0.00 MB
   Apollo.csv: 0.00 MB
   Appollen.csv: 0.00 MB
   Arcfox.csv: 0.0

/kaggle/working/prepared_dataset.zip


🎉 DATA PREPARATION COMPLETE

Next step: run the TRAINING cell. It will load from:
  /kaggle/working/prepared_data/X_train_raw.parquet
  /kaggle/working/prepared_data/y_train_log.npy
  ...etc

Ask for the training code when you're ready.



In [5]:
# =============================================================================
#  CATBOOST-ONLY TRAINING PIPELINE
#  Loads prepared_data/ → trains CatBoost → auto-eval → auto-score → zip → download
#  Kaggle T4x2 compatible
# =============================================================================

import os, sys, json, time, pickle, zipfile, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import FileLink, display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)

# =============================================================================
# SECTION 1: SETUP
# =============================================================================
print("=" * 70)
print("SECTION 1: SETUP")
print("=" * 70)

WORK_DIR = '/kaggle/working'
PREP_DIR = os.path.join(WORK_DIR, 'prepared_data')
OUT_DIR  = os.path.join(WORK_DIR, 'catboost_output')
os.makedirs(OUT_DIR, exist_ok=True)

def _detect_gpu():
    if shutil.which('nvidia-smi') is None:
        return False, 0
    try:
        r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=5)
        if r.returncode == 0:
            return True, max(r.stdout.count('GPU '), 1)
    except Exception:
        pass
    return False, 0

HAS_GPU, N_GPUS = _detect_gpu()
CB_KW = {'task_type': 'GPU', 'devices': '0:1' if N_GPUS >= 2 else '0'} if HAS_GPU else {'task_type': 'CPU'}
print(f"GPUs: {N_GPUS} | CatBoost: {CB_KW['task_type']}")

from catboost import CatBoostRegressor
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error,
                             median_absolute_error, mean_absolute_percentage_error)

# =============================================================================
# SECTION 2: LOAD PREPARED DATA
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 2: LOAD PREPARED DATA")
print("=" * 70)

X_train_raw = pd.read_parquet(os.path.join(PREP_DIR, 'X_train_raw.parquet'))
X_val_raw   = pd.read_parquet(os.path.join(PREP_DIR, 'X_val_raw.parquet'))
X_test_raw  = pd.read_parquet(os.path.join(PREP_DIR, 'X_test_raw.parquet'))

y_train_log  = np.load(os.path.join(PREP_DIR, 'y_train_log.npy'))
y_val_log    = np.load(os.path.join(PREP_DIR, 'y_val_log.npy'))
y_train_orig = np.load(os.path.join(PREP_DIR, 'y_train_orig.npy'))
y_val_orig   = np.load(os.path.join(PREP_DIR, 'y_val_orig.npy'))
y_test_orig  = np.load(os.path.join(PREP_DIR, 'y_test_orig.npy'))
india_test_mask = np.load(os.path.join(PREP_DIR, 'india_test_mask.npy'))

test_sources = pd.read_parquet(os.path.join(PREP_DIR, 'test_sources.parquet'))

print(f"✅ Train: {X_train_raw.shape} | Val: {X_val_raw.shape} | Test: {X_test_raw.shape}")
print(f"✅ India test rows: {india_test_mask.sum()}")

# India sample weights
if 'is_india' in X_train_raw.columns:
    is_india_train = X_train_raw['is_india'].values == 1
else:
    is_india_train = np.zeros(len(X_train_raw), dtype=bool)
sample_weights_train = np.where(is_india_train, 8.0, 1.0)
print(f"✅ India train rows: {is_india_train.sum()} | sample weight: 8×")

# =============================================================================
# SECTION 3: TRAIN CATBOOST
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 3: TRAINING CATBOOST")
print("=" * 70)

model = CatBoostRegressor(
    iterations=12000,
    learning_rate=0.02,
    depth=10,
    l2_leaf_reg=2.0,
    random_seed=42,
    verbose=200,                   # print every 200 iters
    early_stopping_rounds=300,
    loss_function='RMSE',
    **CB_KW
)

start = time.time()
model.fit(
    X_train_raw, y_train_log,
    sample_weight=sample_weights_train,
    eval_set=[(X_val_raw, y_val_log)],
    verbose=200,
)
elapsed = time.time() - start
print(f"\n⏱️  Training time: {elapsed:.1f}s | best_iteration: {model.get_best_iteration()}")

# =============================================================================
# SECTION 4: EVALUATE
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 4: EVALUATION")
print("=" * 70)

# Inverse log transform
y_val_pred  = np.clip(np.expm1(model.predict(X_val_raw)), 0, None)
y_test_pred = np.clip(np.expm1(model.predict(X_test_raw)), 0, None)

def full_metrics(y_true, y_pred):
    return {
        'R²':     r2_score(y_true, y_pred),
        'MAE':    mean_absolute_error(y_true, y_pred),
        'RMSE':   np.sqrt(mean_squared_error(y_true, y_pred)),
        'MedAE':  median_absolute_error(y_true, y_pred),
        'MAPE_%': mean_absolute_percentage_error(y_true, y_pred) * 100,
    }

val_m  = full_metrics(y_val_orig,  y_val_pred)
test_m = full_metrics(y_test_orig, y_test_pred)
india_m    = full_metrics(y_test_orig[india_test_mask],  y_test_pred[india_test_mask]) \
             if india_test_mask.sum() > 1 else {'R²': np.nan, 'MAE': np.nan}
nonindia_m = full_metrics(y_test_orig[~india_test_mask], y_test_pred[~india_test_mask])

print(f"\nVal   R²={val_m['R²']:.4f} | MAE={val_m['MAE']:.2f} | RMSE={val_m['RMSE']:.2f}")
print(f"Test  R²={test_m['R²']:.4f} | MAE={test_m['MAE']:.2f} | RMSE={test_m['RMSE']:.2f}")
print(f"India R²={india_m['R²']:.4f} | MAE={india_m['MAE']:.2f}")
print(f"Non-India R²={nonindia_m['R²']:.4f} | MAE={nonindia_m['MAE']:.2f}")

# Per-source
source_rows = []
for src in pd.unique(test_sources['data_source']):
    mask = (test_sources['data_source'].values == src)
    if mask.sum() < 10:
        continue
    mm = full_metrics(y_test_orig[mask], y_test_pred[mask])
    mm['Source'] = src; mm['n'] = int(mask.sum())
    source_rows.append(mm)
per_source = (pd.DataFrame(source_rows)
              .set_index('Source')[['n', 'R²', 'MAE', 'RMSE', 'MAPE_%']]
              .sort_values('R²', ascending=False))
print("\nPER-SOURCE:")
print(per_source.round(4).to_string())

# Tolerance
residuals = y_test_orig - y_test_pred
abs_err = np.abs(residuals)
print(f"\nResiduals: mean={residuals.mean():+.2f} std={residuals.std():.2f}")
for tol in [5, 10, 20, 50]:
    print(f"   ±{tol:2d} g/km: {(abs_err <= tol).mean()*100:5.1f}%")

# =============================================================================
# SECTION 5: FEATURE IMPORTANCE
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 5: FEATURE IMPORTANCE (top 20)")
print("=" * 70)

fi = pd.DataFrame({
    'feature':    X_train_raw.columns,
    'importance': model.get_feature_importance(),
}).sort_values('importance', ascending=False)
print(fi.head(20).to_string(index=False))

# =============================================================================
# SECTION 6: PLOTS
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 6: PLOTS")
print("=" * 70)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

axes[0,0].scatter(y_test_orig, y_test_pred, alpha=0.3, s=6, color='steelblue')
lims = [0, max(y_test_orig.max(), y_test_pred.max())]
axes[0,0].plot(lims, lims, 'r--', lw=2)
axes[0,0].set_xlabel('Actual CO₂ (g/km)'); axes[0,0].set_ylabel('Predicted CO₂ (g/km)')
axes[0,0].set_title(f'Actual vs Predicted (R²={test_m["R²"]:.4f})', fontweight='bold')
axes[0,0].grid(alpha=0.3)

axes[0,1].scatter(y_test_pred, residuals, alpha=0.3, s=6, color='coral')
axes[0,1].axhline(0, color='r', linestyle='--', lw=2)
axes[0,1].set_xlabel('Predicted'); axes[0,1].set_ylabel('Residual')
axes[0,1].set_title('Residuals', fontweight='bold'); axes[0,1].grid(alpha=0.3)

axes[0,2].hist(residuals, bins=60, color='mediumseagreen', edgecolor='black', alpha=0.7)
axes[0,2].axvline(0, color='r', linestyle='--', lw=2)
axes[0,2].set_xlabel('Residual'); axes[0,2].set_ylabel('Frequency')
axes[0,2].set_title('Residual Distribution', fontweight='bold'); axes[0,2].grid(alpha=0.3)

ps = per_source.sort_values('R²')
colors = ['crimson' if r < 0.5 else 'orange' if r < 0.85 else 'forestgreen' for r in ps['R²']]
axes[1,0].barh(ps.index, ps['R²'], color=colors)
axes[1,0].axvline(0.90, color='blue', linestyle='--', lw=1, label='0.90')
axes[1,0].set_xlabel('R²'); axes[1,0].set_title('Per-Source R²', fontweight='bold')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3, axis='x')

top_fi = fi.head(15).iloc[::-1]
axes[1,1].barh(top_fi['feature'], top_fi['importance'], color='steelblue')
axes[1,1].set_xlabel('Importance'); axes[1,1].set_title('Top-15 Feature Importance', fontweight='bold')
axes[1,1].grid(alpha=0.3, axis='x'); plt.setp(axes[1,1].get_yticklabels(), fontsize=8)

labels, errs = [], []
for src in pd.unique(test_sources['data_source']):
    mask = (test_sources['data_source'].values == src)
    if mask.sum() < 50:
        continue
    labels.append(str(src)[:15]); errs.append(residuals[mask])
if errs:
    axes[1,2].boxplot(errs, labels=labels)
    axes[1,2].axhline(0, color='r', linestyle='--', lw=1)
    axes[1,2].set_ylabel('Residual'); axes[1,2].set_title('Residuals by Source', fontweight='bold')
    plt.setp(axes[1,2].get_xticklabels(), rotation=45, ha='right', fontsize=8)
    axes[1,2].grid(alpha=0.3)

plt.tight_layout()
PLOT_PATH = os.path.join(OUT_DIR, 'evaluation.png')
plt.savefig(PLOT_PATH, dpi=120, bbox_inches='tight')
plt.close()
print(f"✅ Saved {PLOT_PATH}")

# =============================================================================
# SECTION 7: SAVE ARTIFACTS
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 7: SAVE ARTIFACTS")
print("=" * 70)

# CatBoost native format
CBM_PATH = os.path.join(OUT_DIR, 'best_model.cbm')
model.save_model(CBM_PATH)
print(f"✅ {CBM_PATH} ({os.path.getsize(CBM_PATH)/1024/1024:.2f} MB)")

# Pickle (bundle with metadata)
PT_PATH = os.path.join(OUT_DIR, 'best_model.pt')
with open(PT_PATH, 'wb') as f:
    pickle.dump({
        'model':            model,
        'feature_cols':     list(X_train_raw.columns),
        'target_transform': 'log1p (invert with expm1)',
        'target_units':     'g/km CO2',
        'val_R2':           float(val_m['R²']),
        'test_R2':          float(test_m['R²']),
        'india_R2':         float(india_m['R²']) if not np.isnan(india_m['R²']) else None,
        'best_iteration':   int(model.get_best_iteration()),
        'gpu_used':         bool(HAS_GPU),
    }, f, protocol=4)
print(f"✅ {PT_PATH} ({os.path.getsize(PT_PATH)/1024/1024:.2f} MB)")

# CSVs
pd.DataFrame([val_m]).to_csv(os.path.join(OUT_DIR, 'val_metrics.csv'), index=False)
pd.DataFrame([test_m]).to_csv(os.path.join(OUT_DIR, 'test_metrics.csv'), index=False)
per_source.to_csv(os.path.join(OUT_DIR, 'test_per_source.csv'))
fi.to_csv(os.path.join(OUT_DIR, 'feature_importance.csv'), index=False)
pd.DataFrame({
    'Actual':    y_test_orig,
    'Predicted': y_test_pred,
    'Residual':  residuals,
    'Source':    test_sources['data_source'].values,
    'Is_India':  india_test_mask.astype(int),
}).to_csv(os.path.join(OUT_DIR, 'test_predictions.csv'), index=False)

# Metadata
run_meta = {
    'model':           'CatBoost',
    'gpu_used':        bool(HAS_GPU),
    'best_iteration':  int(model.get_best_iteration()),
    'training_time_s': round(elapsed, 1),
    'val_R2':          float(val_m['R²']),
    'test_R2':         float(test_m['R²']),
    'india_R2':        float(india_m['R²']) if not np.isnan(india_m['R²']) else None,
    'nonindia_R2':     float(nonindia_m['R²']),
    'test_MAE':        float(test_m['MAE']),
    'test_RMSE':       float(test_m['RMSE']),
    'within_20':       float((abs_err <= 20).mean()),
    'n_features':      int(X_train_raw.shape[1]),
    'train_rows':      int(len(X_train_raw)),
    'val_rows':        int(len(X_val_raw)),
    'test_rows':       int(len(X_test_raw)),
}
with open(os.path.join(OUT_DIR, 'run_metadata.json'), 'w') as f:
    json.dump(run_meta, f, indent=2)

# Report
summary = f"""
=============================================================
CATBOOST — FINAL REPORT
=============================================================
GPU used      : {HAS_GPU}
Best iteration: {model.get_best_iteration()}
Train time    : {elapsed:.1f}s

Val  R²       : {val_m['R²']:.4f} | MAE {val_m['MAE']:.2f} | RMSE {val_m['RMSE']:.2f}
Test R²       : {test_m['R²']:.4f} | MAE {test_m['MAE']:.2f} | RMSE {test_m['RMSE']:.2f}
India R²      : {india_m['R²']:.4f} | MAE {india_m['MAE']:.2f}
Non-India R²  : {nonindia_m['R²']:.4f} | MAE {nonindia_m['MAE']:.2f}
Within ±20    : {(abs_err <= 20).mean()*100:.1f}%

Per-source:
{per_source.round(4).to_string()}
=============================================================
"""
with open(os.path.join(OUT_DIR, 'final_report.txt'), 'w') as f:
    f.write(summary)
print(summary)

# =============================================================================
# SECTION 8: AUTO-ZIP + AUTO-DOWNLOAD
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 8: ZIP + DOWNLOAD")
print("=" * 70)

ZIP_PATH = os.path.join(WORK_DIR, 'catboost_model_bundle.zip')
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in ['best_model.cbm', 'best_model.pt',
               'val_metrics.csv', 'test_metrics.csv', 'test_per_source.csv',
               'test_predictions.csv', 'feature_importance.csv',
               'run_metadata.json', 'final_report.txt', 'evaluation.png']:
        fp = os.path.join(OUT_DIR, fn)
        if os.path.exists(fp):
            zf.write(fp, fn)

size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"✅ ZIP: {ZIP_PATH} ({size_mb:.2f} MB)")

print("\n" + "=" * 70)
print("🎉 DONE")
print("=" * 70)
print(f"Test R²  = {test_m['R²']:.4f}")
print(f"India R² = {india_m['R²']:.4f}")
print(f"Test MAE = {test_m['MAE']:.2f} g/km")
print("\nBundle contains: best_model.cbm (native), best_model.pt (pickle),")
print("4 CSVs, feature importance, report, plot")
print("\n📥 Download below 👇\n")
display(FileLink('catboost_model_bundle.zip'))

SECTION 1: SETUP
GPUs: 2 | CatBoost: GPU

SECTION 2: LOAD PREPARED DATA
✅ Train: (82630, 57) | Val: (14582, 57) | Test: (24304, 57)
✅ India test rows: 2000
✅ India train rows: 6800 | sample weight: 8×

SECTION 3: TRAINING CATBOOST
0:	learn: 0.5038287	test: 0.4163168	best: 0.4163168 (0)	total: 31.1ms	remaining: 6m 13s
200:	learn: 0.3550371	test: 0.1985134	best: 0.1985134 (200)	total: 3.61s	remaining: 3m 31s
400:	learn: 0.2813759	test: 0.1969532	best: 0.1969532 (400)	total: 7.18s	remaining: 3m 27s
600:	learn: 0.2267215	test: 0.1962991	best: 0.1962991 (600)	total: 10.7s	remaining: 3m 23s
800:	learn: 0.1852296	test: 0.1950736	best: 0.1950684 (798)	total: 14.3s	remaining: 3m 19s
1000:	learn: 0.1541744	test: 0.1935768	best: 0.1935655 (995)	total: 17.9s	remaining: 3m 17s
1200:	learn: 0.1319558	test: 0.1918232	best: 0.1918232 (1200)	total: 21.6s	remaining: 3m 13s
1400:	learn: 0.1149849	test: 0.1901370	best: 0.1901370 (1400)	total: 25.1s	remaining: 3m 10s
1600:	learn: 0.1022061	test: 0.1889522	

/kaggle/working/catboost_model_bundle.zip